# 02 — Diabetes Risk Modelling

**P.U.L.S.E.**

Trains three XGBoost classifier variants on diabetes data and evaluates them:

| Model | Dataset | Notes |
|-------|---------|-------|
| V1 | D130 + NHANES combined | baseline |
| V2a | D130 only | readmission labels |
| V2b | NHANES only | clinical HbA1c labels — best AUC (0.97) |

Covers: feature selection · PR-curve threshold · SHAP interpretability · MLflow tracking

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_curve, RocCurveDisplay
import shap

sns.set_theme(style='whitegrid', palette='muted')
print('Environment ready')

## 1  Load feature store & NHANES parquets

In [ ]:
DATA = os.path.join(ROOT, 'data', 'processed')

master    = pd.read_parquet(os.path.join(DATA, 'master_patient_table.parquet'))
nhanes_15 = pd.read_parquet(os.path.join(DATA, 'nhanes_2015.parquet'))
nhanes_21 = pd.read_parquet(os.path.join(DATA, 'nhanes_2021.parquet'))

print('Master:   ', master.shape)
print('NHANES-15:', nhanes_15.shape)
print('NHANES-21:', nhanes_21.shape)

## 2  Run V2 model suite (trains + evaluates all three variants)

In [ ]:
from src.models.diabetes_model_v2 import run_all_v2

results = run_all_v2()
for name, m in results.items():
    print(f'{name}: AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}  threshold={m["threshold"]:.3f}')

## 3  ROC curves — all three variants

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colours = ['#1976D2', '#388E3C', '#D32F2F']
for (name, m), colour in zip(results.items(), colours):
    RocCurveDisplay(
        fpr=m['fpr'], tpr=m['tpr'],
        roc_auc=m['auc'],
        estimator_name=name,
    ).plot(ax=ax, color=colour)

ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_title('ROC Curves — Diabetes Models')
plt.tight_layout()
plt.show()

## 4  Precision-Recall curve — V2b-NHANES (production model)

In [ ]:
import pickle
MODEL_DIR = os.path.join(ROOT, 'models')

with open(os.path.join(MODEL_DIR, 'nhanes_diab_xgb.pkl'), 'rb') as f:
    model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'nhanes_diab_features.pkl'), 'rb') as f:
    features = pickle.load(f)

nhanes_labelled = nhanes_21.dropna(subset=['diabetes_label'])
X_test = nhanes_labelled[features]
y_test = nhanes_labelled['diabetes_label'].astype(int)

proba = model.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec, prec, color='#D32F2F', lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'PR Curve — V2b-NHANES  (AUC={roc_auc_score(y_test, proba):.4f})')
plt.tight_layout()
plt.show()

## 5  SHAP feature importance — V2b-NHANES

In [ ]:
explainer = shap.TreeExplainer(model.get_booster())
shap_vals  = explainer.shap_values(X_test.sample(500, random_state=42))

shap.summary_plot(
    shap_vals,
    X_test.sample(500, random_state=42),
    plot_type='bar',
    show=False,
)
plt.title('SHAP Feature Importance — Diabetes V2b-NHANES')
plt.tight_layout()
plt.show()

## 6  Calibration & threshold sensitivity

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import f1_score

# Calibration
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

CalibrationDisplay.from_predictions(
    y_test, proba, n_bins=10, ax=ax[0], name='V2b-NHANES'
)
ax[0].set_title('Calibration')

# F1 vs threshold
thresholds = np.linspace(0.1, 0.9, 80)
f1s = [f1_score(y_test, (proba >= t).astype(int), zero_division=0) for t in thresholds]
ax[1].plot(thresholds, f1s, color='#1976D2', lw=2)
ax[1].axvline(thresholds[np.argmax(f1s)], color='red', linestyle='--', label=f'opt @{thresholds[np.argmax(f1s)]:.2f}')
ax[1].set_xlabel('Decision threshold')
ax[1].set_ylabel('F1 score')
ax[1].set_title('F1 vs threshold')
ax[1].legend()

plt.suptitle('Model Evaluation — V2b-NHANES')
plt.tight_layout()
plt.show()